In [5]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import ConvLSTM1D, Flatten, Dense
from pyswarms.single.global_best import GlobalBestPSO

os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

class WeightLogger(tf.keras.callbacks.Callback):
    def __init__(self, layer_index=0):
        self.layer_index = layer_index
        self.weights_per_epoch = []

    def on_epoch_end(self, epoch, logs=None):
        weights = self.model.layers[self.layer_index].get_weights()[0]
        self.weights_per_epoch.append(weights.copy())

def load_and_prepare_data(csv_path, production_column='production', window_size=21):
    df = pd.read_csv(csv_path, sep=',')
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(df.values)
    target_scaler = MinMaxScaler()
    target_scaler.fit(df[[production_column]])
    target_col_idx = df.columns.get_loc(production_column)
    x, y = [], []
    for i in range(window_size, len(data_scaled)):
        x.append(data_scaled[i-window_size:i])
        y.append(data_scaled[i, target_col_idx])
    x, y = np.array(x), np.array(y)
    train_split_index = int(0.8 * len(x))
    test_split_index = int(0.9 * len(x))
    x_train, y_train = x[:train_split_index], y[:train_split_index]
    x_test, y_test = x[train_split_index:test_split_index], y[train_split_index:test_split_index]
    x_val, y_val = x[test_split_index:], y[test_split_index:]
    x_train_conv = np.expand_dims(x_train, axis=2)
    x_test_conv = np.expand_dims(x_test, axis=2)
    x_val_conv = np.expand_dims(x_val, axis=2)
    return x_train_conv, y_train, x_test_conv, y_test, x_val_conv, y_val, df, target_scaler

def build_convlstm_model(lr, filters1, filters2, dense_units, input_shape):
    model = Sequential([
        ConvLSTM1D(filters=int(filters1), kernel_size=1, activation='tanh', return_sequences=True, input_shape=input_shape),
        ConvLSTM1D(filters=int(filters2), kernel_size=1, activation='tanh', return_sequences=False),
        Flatten(),
        Dense(int(dense_units), activation='relu'),
        Dense(1, activation="linear")
    ])
    model.compile(optimizer=Adam(learning_rate=lr), loss="mae")
    return model

def train_and_evaluate_model(model, x_train, y_train, x_val, y_val, epochs=30, batch_size=256, dataset_name="dataset"):
    stop_early = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=epochs,
                        batch_size=batch_size,
                        verbose=0,
                        callbacks=[stop_early])
    return history

def inference_and_plot(model, x_test, y_test, target_scaler, dataset_name):
    preds = model.predict(x_test)
    y_test_real = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_real = target_scaler.inverse_transform(preds.reshape(-1, 1)).flatten()
    mae = mean_absolute_error(y_test_real, y_pred_real)
    mse = mean_squared_error(y_test_real, y_pred_real)
    r2 = r2_score(y_test_real, y_pred_real)
    return mae, mse, r2

def run_experiment_with_pso(csv_path, dataset_name):
    x_train, y_train, x_test, y_test, x_val, y_val, df, target_scaler = load_and_prepare_data(csv_path)
    input_shape = x_train.shape[1:]

    def pso_objective_function(hyperparams_array):
        results = []
        for params in hyperparams_array:
            lr = 10 ** params[0]
            filters1 = int(params[1])
            filters2 = int(params[2])
            dense_units = int(params[3])
            model = build_convlstm_model(lr, filters1, filters2, dense_units, input_shape)
            history = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                                epochs=200, batch_size=256, verbose=0)
            val_loss = min(history.history['val_loss'])
            results.append(val_loss)
        return np.array(results)

    bounds = (np.array([np.log10(1e-4), 32, 32, 32]),
              np.array([np.log10(1e-2), 128, 128, 128]))
    
    optimizer = GlobalBestPSO(n_particles=10, dimensions=4,
                              options={'c1': 0.5, 'c2': 0.3, 'w': 0.9},
                              bounds=bounds)
    
    best_cost, best_pos = optimizer.optimize(pso_objective_function, iters=5)

    best_lr = 10 ** best_pos[0]
    best_filters1 = int(best_pos[1])
    best_filters2 = int(best_pos[2])
    best_dense_units = int(best_pos[3])

    print(f"\nBest hyperparameters from PSO:\nLR: {best_lr}\nFilters1: {best_filters1}\nFilters2: {best_filters2}\nDense: {best_dense_units}")

    model = build_convlstm_model(best_lr, best_filters1, best_filters2, best_dense_units, input_shape)
    train_and_evaluate_model(model, x_train, y_train, x_val, y_val, dataset_name=dataset_name)
    mae, mse, r2 = inference_and_plot(model, x_test, y_test, target_scaler, dataset_name)
    print(f"{dataset_name} — MAE: {mae:.4f}, MSE: {mse:.4f}, R²: {r2:.4f}")

if __name__ == "__main__":
    run_experiment_with_pso("../DataCleaning/scaled_dataset_clean_sansBatiment5.csv", "scaled_dataset_clean_sansbatiment5")


2025-07-04 11:39:54,299 - pyswarms.single.global_best - INFO - Optimize for 5 iters with {'c1': 0.5, 'c2': 0.3, 'w': 0.9}
pyswarms.single.global_best:   0%|          |0/5c:\Users\Natha\anaconda3\envs\IR_pv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
pyswarms.single.global_best:  20%|██        |1/5, best_cost=0.02


KeyboardInterrupt: 